# 03 — Async / Await

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/03_async_and_await.ipynb)

## 📌 What is it?
`asyncio` lets Python run concurrent I/O-bound tasks on a single thread using an event loop. `async def` defines coroutines; `await` pauses execution until the awaited task completes.

## 🧠 Why AI Engineers need this
LLM APIs are I/O bound. Calling 100 LLMs in parallel with `asyncio.gather()` is 100x faster than sequential calls. Async is essential for production AI systems.

In [ ]:
import asyncio
import time

# ── COROUTINE BASICS ──
async def fetch_completion(prompt: str, delay: float) -> str:
    """Simulate an async LLM API call."""
    await asyncio.sleep(delay)   # non-blocking wait
    return f"Response to: {prompt[:30]}"

async def main():
    # Sequential — slow!
    start = time.time()
    r1 = await fetch_completion("What is AI?", 0.1)
    r2 = await fetch_completion("What is ML?", 0.1)
    r3 = await fetch_completion("What is DL?", 0.1)
    print(f"Sequential: {time.time()-start:.2f}s")
    print(r1)

asyncio.run(main())

In [ ]:
import asyncio, time

async def fetch_completion(prompt: str, delay: float = 0.1) -> str:
    await asyncio.sleep(delay)
    return f"Response: {prompt[:20]}..."

# ── PARALLEL with asyncio.gather ──
async def main_parallel():
    prompts = [
        ("Explain transformers", 0.1),
        ("What is attention?", 0.15),
        ("Define embeddings", 0.08),
        ("Explain RAG", 0.12),
        ("What is fine-tuning?", 0.09),
    ]
    
    start = time.time()
    
    # All run concurrently!
    results = await asyncio.gather(*[
        fetch_completion(p, d) for p, d in prompts
    ])
    
    elapsed = time.time() - start
    print(f"Parallel: {elapsed:.2f}s for {len(prompts)} calls")
    print(f"Expected sequential: {sum(d for _,d in prompts):.2f}s")
    for r in results:
        print(f"  • {r}")

asyncio.run(main_parallel())

In [ ]:
import asyncio

# ── ASYNC CONTEXT MANAGER ──
class AsyncAPIClient:
    def __init__(self, base_url: str):
        self.base_url = base_url
        self.session = None
    
    async def __aenter__(self):
        print(f"Opening session to {self.base_url}")
        self.session = "mock-session"   # would be aiohttp.ClientSession()
        return self
    
    async def __aexit__(self, *args):
        print("Closing session")
        self.session = None
    
    async def get(self, endpoint: str) -> dict:
        await asyncio.sleep(0.05)   # simulate network
        return {"endpoint": endpoint, "status": "ok"}

async def main():
    async with AsyncAPIClient("https://api.anthropic.com") as client:
        response = await client.get("/v1/messages")
        print(response)

asyncio.run(main())

In [ ]:
import asyncio

# ── ASYNC GENERATOR (streaming!) ──
async def stream_tokens(text: str, delay: float = 0.02):
    """Simulate LLM token streaming."""
    for word in text.split():
        await asyncio.sleep(delay)
        yield word + " "

async def main():
    print("Streaming: ", end="", flush=True)
    async for token in stream_tokens("The quick brown fox jumps"):
        print(token, end="", flush=True)
    print()

asyncio.run(main())

## ⚠️ Common Mistakes
```python
# ❌ Calling async function without await
result = my_coroutine()  # returns a coroutine object, not the result!

# ✅
result = await my_coroutine()

# ❌ Using asyncio.run() inside another coroutine
async def bad():
    asyncio.run(other())  # RuntimeError!

# ✅
async def good():
    await other()

# ❌ CPU-bound tasks in asyncio (blocks the event loop!)
async def bad_cpu():
    result = heavy_computation()  # blocks everything!

# ✅ Use run_in_executor for CPU-bound
async def good_cpu():
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(None, heavy_computation)
```

## 🔗 What's Next?
[04 — Threading & Multiprocessing →](04_multithreading_multiprocessing.ipynb)